<b>Problem 1. My Own CNN Model</b><br>
Design an image classification model class similar to one on page 7 of Lecture Note 13, but at this time the model configuration is given as the list of nodes instead of the number of layers. For example, instead of `nconv=3`, using `conv=(32,32,32)` indicated three convolution layers of which units are 32. Similarly, `conv=(64,32,32)` can make a convolution layer with 64 units followed by two convolution layers with 32 units. Fully connected layer can be also specified by `fc=(32,10)` instead of ndense=2. In other words, the same model on page 7 can be specified by `MyModel(conv=(32,32,32),fc=(32,10))`.<br>
Apply this model for the fashion MNIST dataset and find the best model configuration.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Fashion MNIST 데이터 로드 및 전처리
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# 이미지 정규화 및 채널 차원 추가 (28x28 -> 28x28x1)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# [요구사항 구현] 동적 모델 생성 함수 생성
def MyModel(conv=(32, 32, 32), fc=(32, 10)):
    model = models.Sequential()

    # 1. 입력층 정의
    model.add(layers.Input(shape=(28, 28, 1)))

    # 2. Convolution 레이어 동적 추가
    for filters in conv:
        model.add(layers.Conv2D(filters, (3, 3), padding='same', activation='relu'))
        model.add(layers.MaxPooling2D((2, 2), padding='same')) # 크기 축소용 풀링층

    # 3. Fully Connected(Dense) 레이어 전 Flatten
    model.add(layers.Flatten())

    # 4. Dense 레이어 동적 추가 (마지막 레이어 전까지)
    for i in range(len(fc) - 1):
        model.add(layers.Dense(fc[i], activation='relu'))

    # 5. 출력층 추가 (마지막 노드는 클래스 개수 10개 및 Softmax 적용)
    model.add(layers.Dense(fc[-1], activation='softmax'))

    return model

# --- 여러 구조 실험해보기 ---
# 예시 1: 13-7에 제시된 기본 구조 (32,32,32) -> (32,10)
print("=== 구조 1 테스트 ===")
model1 = MyModel(conv=(32, 32, 32), fc=(32, 10))
model1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model1.fit(x_train, y_train, epochs=3, validation_data=(x_test, y_test), batch_size=64)

# 예시 2: 성능 향상 (Best Configuration 탐색)
print("\n=== 구조 2 테스트 (더 깊고 넓은 구조) ===")
model2 = MyModel(conv=(64, 64, 32), fc=(64, 10))
model2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model2.fit(x_train, y_train, epochs=3, validation_data=(x_test, y_test), batch_size=64)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
=== 구조 1 테스트 ===
Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 60s 62ms/step - accuracy: 0.7979 - loss: 0.5583 - val_accuracy: 0.8467 - val_loss: 0.4144
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 80s 59ms/step - accuracy: 0.8705 - loss: 0.3575 - val_accuracy: 0.8706 - val_loss: 0.3536
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 54s 58ms/step - accuracy: 0.8888 - loss: 0.3083 - val_accuracy: 0.8798 - val_loss: 0.3319

=== 구조 2 테스트 (더 깊고 넓은 구조) ===
Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 124s 130ms/step - accuracy: 0.8145 - loss: 0.5094 - val_accuracy: 0.8586 - val_loss: 0.3906
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 150s 139ms/step - accuracy: 0.8847 - loss: 0.3180 - val_accuracy: 0.8887 - val_loss: 0.3041
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 122s 130ms/step - accuracy: 0.9016 - loss: 0.2685 - val_accuracy: 0.8994 - v

구조 1의 정확도 보다 구조 2의 정확도가 높고 (0.88 < 0.90), 구조 1의 검증 손실이 구조 2의 검증 손실보다 높습니다.(0.33 > 0.28)

따라서 모델2가 더 정확하다고 할 수 있습니다.

<b>Problem 2. My Own CNN Model 2</b><br>
The class in problem 1 can be further developed including dropout and normalizaitons. Now we want to use `conv=('32ndp','32ndp','32ndp')` which means three repeats of a convolution layer with 32 units, followed by batch normalization layer (`n`), dropout layer (`d`) and max-pooling layer (`p`). `conv=(32,32,32)` in problem 1 is equivalent to `conv=('32p','32p','32p')`. `conv=('64n','32np')` means a convolution layer with 64 unit and batch normalization, then directly followed by, without pooling, another convolution layer with 32 units with normalization and pooling. There is no dropout. Use the same expression with problem 1 for the FC layer. dropout probability should be specified another argument `dropout`. For example, the model on page 32 of Lecture Note 13 can be expressed `MyModel( conv=('64ndp','64ndp','64ndp'), fc=(64,10), dropout=0.2 )`<br>
Apply this model to Cifar10 dataset and find the best model.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import re

# CIFAR-10 데이터 로드 및 전처리
cifar10 = tf.keras.datasets.cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 이미지 정규화 (CIFAR-10은 이미 3채널 컬러 이미지 32x32x3 형태임)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# [요구사항 구현] 문자열 파싱 기능이 포함된 동적 모델 생성 함수
def MyAdvancedModel(conv=('32ndp', '32ndp', '32ndp'), fc=(32, 10), dropout=0.25):
    model = models.Sequential()

    # 1. 입력층 정의 (CIFAR-10 이미지 크기: 32x32x3)
    model.add(layers.Input(shape=(32, 32, 3)))

    # 2. Convolution 영역 파싱 및 동적 추가
    for layer_cfg in conv:
        # 만약 숫자로만 들어왔다면 기본값인 'p'를 붙여서 처리 (예: 32 -> '32p')
        if isinstance(layer_cfg, int):
            layer_cfg = f"{layer_cfg}p"

        # 정규표현식을 사용해 숫자(필터 수)와 알파벳 기호들을 분리
        filters = int(re.findall(r'\d+', layer_cfg)[0])
        options = re.findall(r'[a-zA-Z]', layer_cfg)

        # 기본 Conv2D 추가
        model.add(layers.Conv2D(filters, (3, 3), padding='same', activation='relu'))

        # 기호 순서대로 레이어 배치
        for opt in options:
            if opt == 'n':
                model.add(layers.BatchNormalization())
            elif opt == 'd':
                model.add(layers.Dropout(dropout))
            elif opt == 'p':
                model.add(layers.MaxPooling2D((2, 2), padding='same'))

    # 3. Fully Connected 영역
    model.add(layers.Flatten())

    for i in range(len(fc) - 1):
        model.add(layers.Dense(fc[i], activation='relu'))
        # FC 영역에도 옵션으로 드롭아웃을 넣어줄 수 있으나 문제 조건에 맞춰 기본 배치

    # 4. 출력층 (CIFAR-10의 클래스 개수는 10개)
    model.add(layers.Dense(fc[-1], activation='softmax'))

    return model

# --- 여러 구조 실험 및 최적의 모델 탐색 ---

# 실험 1: 기본 정규화+드롭아웃+풀링 구조 ('32ndp', '32ndp', '32ndp')
print("=== 구조 1 (정규화 및 드롭아웃 포함 3층 구조) ===")
model_1 = MyAdvancedModel(conv=('32ndp', '32ndp', '32ndp'), fc=(64, 10), dropout=0.2)
model_1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_1.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test), batch_size=64)

# 실험 2: 풀링을 건너뛰며 채널을 넓히는 고급 구조 ('64n', '64np', '128ndp')
print("\n=== 구조 2 (풀링 제어 및 채널 확장 구조) ===")
model_2 = MyAdvancedModel(conv=('64n', '64np', '128ndp'), fc=(128, 10), dropout=0.3)
model_2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_2.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test), batch_size=64)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
=== 구조 1 (정규화 및 드롭아웃 포함 3층 구조) ===
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 131s 161ms/step - accuracy: 0.4646 - loss: 1.4923 - val_accuracy: 0.3991 - val_loss: 1.9610
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 123s 158ms/step - accuracy: 0.6030 - loss: 1.1163 - val_accuracy: 0.5840 - val_loss: 1.1927
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 138s 153ms/step - accuracy: 0.6591 - loss: 0.9656 - val_accuracy: 0.4042 - val_loss: 2.1656
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 138s 148ms/step - accuracy: 0.6918 - loss: 0.8766 - val_accuracy: 0.5864 - val_loss: 1.2717
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 143s 149ms/step - accuracy: 0.7117 - loss: 0.8171 - val_accuracy: 0.5555 - val_loss: 1.4990

=== 구조 2 (풀링 제어 및 채널 확장 구조) ===
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 619s 787ms/step - accuracy: 0.5192 - loss: 1.3822 - val_accuracy: 0.5912 - val_loss: 1.3180
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 631s 800ms/step - accuracy: 0.6950 - loss: 0.8683 - va

1번과 마찬가지로 구조 2 모델이 최종 베스트 모델로 선정되었습니다.

구조 1은 5번째 에포크에서 검증 정확도(val_accuracy) 55.55%, 검증 손실(val_loss) 1.4990을 기록했습니다.

반면, 구조 2는 에포크당 학습 시간은 약 10분으로 더 오래 걸렸으나, 최종 검증 정확도 66.23%, 검증 손실 1.2080을 기록하며 구조 1에 비해 약 11%가량 향상된 분류 성능을 보였습니다.